## Build a Wflow Sediment model

This notebook demonstrates how to prepare a **Wflow Sediment** model using the command line interace (CLI).

All lines in this notebook which start with `!` are executed from the command line. Within the notebook environment the logging messages are shown after completion. You can also copy these lines and paste these in your shell to get more direct feedback.

### HydroMT CLI build interface

Lets first check if the Wflow sediment model is recognized by HydroMT

In [1]:
# this should return "wflow_sbm, wflow_sediment"
# as well as the generic HydroMT model "model"
!hydromt --models

Model plugins:
	- model (hydromt 1.3.0)
	- example_model (hydromt 1.3.0)
	- wflow_sbm (hydromt_wflow 1.0.1)
	- wflow_sediment (hydromt_wflow 1.0.1)


Using the **HydroMT build** API we can setup a complete model from scratch. Let's get an overview of all the available options:

In [ ]:
!hydromt build --help

### Model setup configuration

The HydroMT configuration file (YAML) contains the model setup configuration and determines which methods are used to prepare the different components of a Wflow Sediment model and in which order and optionally sets non-default arguments for each method. This configuration is passed to HydroMT using `-i <path_to_config_file>`. We have prepared several example files which are available in the model repository [examples folder](https://github.com/Deltares/hydromt_wflow/tree/main/examples) and from the [docs (building a model)](https://deltares.github.io/hydromt_wflow/latest/user_guide/sediment_build.html). 

Each header (added as a list item in steps) (e.g. `setup_basemaps:`) corresponds to a model method. All model methods are explained in the [docs (model methods)](https://deltares.github.io/hydromt_wflow/latest/user_guide/sediment_model_setup.html). 

We will load the default wflow sediment build configuration yaml file for inspection:

In [2]:
fn_config = "wflow_sediment_build.yml"
with open(fn_config, "r") as f:
    txt = f.read()
print(txt)

steps:
  - setup_config:                     # options parsed to wflow toml file <section>.<option>
      data:
        time.starttime: "2010-02-02T00:00:00"
        time.endtime: "2010-02-10T00:00:00"
        time.timestepsecs: 86400
        output.netcdf_grid.path: output.nc
        output.netcdf_grid.compressionlevel: 1
        output.netcdf_grid.variables.soil_erosion__mass_flow_rate: soil_loss
        output.netcdf_grid.variables.river_water_sediment__suspended_mass_concentration: suspended_solids

  - setup_basemaps:
      hydrography_fn: merit_hydro      # source hydrography data {merit_hydro, merit_hydro_1k}
      basin_index_fn: merit_hydro_index # source of basin index corresponding to hydrography_fn
      upscale_method: ihu              # upscaling method for flow direction data, by default 'ihu'
      res: 0.00833           # build the model at a 30 arc sec (~1km) resolution
      region:
        subbasin: [12.2051, 45.8331]
        strord: 4
        bounds: [11.70, 45.35,

### Setup Wflow Sediment model from scratch

In [3]:
# NOTE: copy this line (without !) to your shell for more direct feedback
# NOTE: the region argument has been removed since hydromt v1.0.0 and can now only be configured from the configuration file under the setup_basemap header
!hydromt build wflow_sediment "./wflow_test_sediment" -i wflow_sediment_build.yml -d artifact_data --fo -v

2026-03-09 10:14:15,756 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 10:14:16,166 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Reading data catalog artifact_data latest
2026-03-09 10:14:16,167 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\.hydromt\artifact_data\v1.0.0\data_catalog.yml
2026-03-09 10:14:16,936 - hydromt.model.model - model - INFO - Initializing wflow_sediment model from hydromt_wflow (v1.0.1).
2026-03-09 10:14:16,936 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 10:14:16,957 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 10:14:16,958 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Users/matti/miniconda3/envs/hydromt-wflow/Lib/site-packages/

The example above means the following: run **hydromt build** with:

* `wflow_sediment` : i.e. build a wflow sediment model
* `./wflow_test_sediment` : output model folder
* `-i wflow_sediment_build.yml` : model setup configuration file
* `-d artifact_data` : use pre-defined artifact_data catalog.
* `-v` : give some extra verbosity (1 * v) to display feedback on screen. Now INFO messages are provided.


Next we check which files have been created. The model root should contain two netcdf (.nc) files, one for the staticmaps and one for the forcing data, a wflow run configuration (.toml) file. These files are sufficient to run the wflow sediment model. In addition several geometry (.geojson) files are saved in the *staticgeoms* folder.  Finally, the setup process is logged in the hydromt.log file. 

In [4]:
import os

root = "wflow_test_sediment"
for path, _, files in os.walk(root):
    print(path)
    for name in files:
        if name.endswith(".xml"):
            continue
        print(f" - {name}")

wflow_test_sediment
 - hydromt.log
 - hydromt_data.yml
 - staticmaps.nc
 - wflow_sediment.toml
wflow_test_sediment\staticgeoms
 - basins.geojson
 - gauges_grdc.geojson
 - meta_basins_highres.geojson
 - meta_natural_reservoirs.geojson
 - outlets.geojson
 - region.geojson
 - reservoirs.geojson
 - rivers.geojson


### Visualize and/or inspect model schematization

You can (copy and) adapt the following example notebooks in order to visualize your Wflow Sediment model: 

* The **wflow plot** example notebook contains scripts to visualize your model
* The **wflow nc to raster files** example notebook contains scripts to write the nc model files to a netcdf which can be used to inspect (and modify) the model layers in e.g. QGIS.

### Add Wflow Sediment layers to an existing Wflow model

If you already have a hydrological **Wflow** model and you want to extend it in order to include **sediment** as well, then you do not need to build the wflow_sediment from scratch. You can instead **update** the wflow model with the additional components needed by wflow_sediment. 

These components are available in the template **wflow_extend_sediment.yml**:

In [5]:
fn_config = "wflow_extend_sediment.yml"
with open(fn_config, "r") as f:
    txt = f.read()
print(txt)

steps:
  - config.read: # read a template config file for wflow sediment to be able to extend from a sbm model
      filename: data/wflow_sediment.toml

  - setup_config: # options parsed to wflow toml file <section>.<option>
      data:
        time.starttime: 2010-01-01T00:00:00
        time.endtime: 2010-03-31T00:00:00
        time.timestepsecs: 86400
        output.netcdf_grid.path: output.nc
        output.netcdf_grid.compressionlevel: 1
        output.netcdf_grid.variables.soil_erosion__mass_flow_rate: soil_loss
        output.netcdf_grid.variables.river_water_sediment__suspended_mass_concentration: suspended_solids

  - setup_riverbedsed:
      bedsed_mapping_fn:       # path to a mapping csv file from streamorder to river bed particles characteristics if any, else default is used

  # If you do have reservoirs, you can re-run setup_natural_reservoirs and setup_reservoirs to add it
  - setup_reservoirs:
      reservoirs_fn: hydro_reservoirs  # source for reservoirs shape and att

Note that because we are starting from a sbm model, we start here with a ``read_config`` step in order to get a default **wflow_sediment.toml** that contains the minimum config options of the sediment model including variables names of the already present maps (eg slope, river mask...). You may have to modify this template if the staticmaps name do not correspond to the names in your own wflow sbm model.

Let's update the hydrological **Wflow** model available from our example for the Piave subbasin:

In [6]:
# NOTE: copy this line (without !) to your shell for more direct feedback
!hydromt update wflow_sediment "wflow_piave_subbasin" -o "./wflow_test_extend_sediment" -i wflow_extend_sediment.yml -d artifact_data --fo -v

2026-03-09 10:20:21,799 - hydromt - log - INFO - HydroMT version: 1.3.0
2026-03-09 10:20:22,224 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Reading data catalog artifact_data latest
2026-03-09 10:20:22,224 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\.hydromt\artifact_data\v1.0.0\data_catalog.yml
2026-03-09 10:20:23,045 - hydromt.model.model - model - INFO - Initializing wflow_sediment model from hydromt_wflow (v1.0.1).
2026-03-09 10:20:23,045 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 10:20:23,066 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 10:20:23,066 - hydromt.hydromt_wflow.components.config - config - WARNING - No config was found at C:/Data/TUD/MSc_CE/Courses/2nd_year/7.CIE5060_Thesis/Codes/MSc_thesis_

The example above means the following: run **hydromt update** with:

* `wflow_sediment` : i.e. update a wflow_sediment model (in our case a wflow model but with wflow_sediment components)
* `wflow_piave_subbasin` : hydrological wflow model folder
* `-o "./wflow_test_extend_sediment"`: output combined wflow hydrology+sediment models
* `-i wflow_extend_sediment.yml` : setup configuration file containing wflow sediment specific components
* `-d artifact_data` : use the pre-defined artifact_data catalog
* `-v` : give some extra verbosity (1 * v) to display feedback on screen. Now INFO messages are provided.